In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "India"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [4]:
location = location.title()

In [5]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2021        2022        7.684611e+04
                  0.019178   0.076712    2021        2022        2.268013e+05
                  0.076712   0.500000    2021        2022        1.653024e+06
                  0.500000   1.000000    2021        2022        1.904590e+06
                  1.000000   2.000000    2021        2022        3.736238e+06
                  2.000000   5.000000    2021        2022        1.079625e+07
                  5.000000   10.000000   2021        2022        1.693204e+07
                  10.000000  15.000000   2021        2022        1.577085e+07
                  15.000000  20.000000   2021        2022        1.379964e+07
                  20.000000  25.000000   2021        2022        1.128166e+07
                  25.000000  30.000000   2021        2022        9.287220e+06
                  30.000000  35.000000   2021        2022        7.471066e+06
  

In [6]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
Nigeria   Female  10.0       15.0     2021        2022      lower_value    0.001521
                                                            mean_value     0.003104
                                                            upper_value    0.006026
                  15.0       20.0     2021        2022      lower_value    0.067972
                                                            mean_value     0.077594
                                                            upper_value    0.088759
                  20.0       25.0     2021        2022      lower_value    0.181182
                                                            mean_value     0.202959
                                                            upper_value    0.223959
                  25.0       30.0     2021        2022      lower_value    0.208116
                                                            mean_value     0.221416
    

In [7]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2021        2022        0.000000
                  0.019178   0.076712    2021        2022        0.000000
                  0.076712   0.500000    2021        2022        0.000000
                  0.500000   1.000000    2021        2022        0.000000
                  1.000000   2.000000    2021        2022        0.000000
                  2.000000   5.000000    2021        2022        0.000000
                  5.000000   10.000000   2021        2022        0.000000
                  10.000000  15.000000   2021        2022        0.003104
                  15.000000  20.000000   2021        2022        0.077594
                  20.000000  25.000000   2021        2022        0.202959
                  25.000000  30.000000   2021        2022        0.221416
                  30.000000  35.000000   2021        2022        0.202088
                  35.000000  40.000000   2021     

In [8]:
births = pop * asfr
births[births > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2021        2022        4.895440e+04
                  15.0       20.0     2021        2022        1.070775e+06
                  20.0       25.0     2021        2022        2.289711e+06
                  25.0       30.0     2021        2022        2.056344e+06
                  30.0       35.0     2021        2022        1.509810e+06
                  35.0       40.0     2021        2022        8.395243e+05
                  40.0       45.0     2021        2022        3.610966e+05
                  45.0       50.0     2021        2022        1.455784e+05
                  50.0       55.0     2021        2022        1.149517e+04
Name: value, dtype: float64

In [9]:
births = births.sum()
f"{int(births):,}"

'8,333,288'

In [10]:
sim_baseline_births = pd.read_parquet(
    f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet"
)
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,1,zero,82,0,8.0
1,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,2,zero,82,0,5.0
2,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,3,zero,82,0,6.0
3,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,4,zero,82,0,4.0
4,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,5,zero,82,0,3.0
...,...,...,...,...,...,...,...,...,...,...,...
809995,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,1,baseline,63,0,0.0
809996,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,2,baseline,63,0,0.0
809997,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,3,baseline,63,0,0.0
809998,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,4,baseline,63,0,0.0


In [11]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == "baseline")
    & (sim_baseline_births.sub_entity == "live_birth")
]
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
16205,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,1,baseline,112,0,49.0
16206,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,2,baseline,112,0,41.0
16207,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,3,baseline,112,0,43.0
16208,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,4,baseline,112,0,43.0
16209,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,5,baseline,112,0,38.0
...,...,...,...,...,...,...,...,...,...,...,...
809990,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,1,baseline,63,0,0.0
809991,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,2,baseline,63,0,0.0
809992,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,3,baseline,63,0,0.0
809993,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,4,baseline,63,0,0.0


In [12]:
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

7860828.0

In [13]:
births

8333288.405715805

In [14]:
scalar = births / sim_baseline_births
scalar

1.060103134900777

In [15]:
for result in ["ylds", "ylls", "deaths", "person_time"]:
    df = pd.read_parquet(
        f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)